In [1]:
!apt-get update
!apt-get install -y swig
!pip install box2d-py
!pip install "gymnasium[box2d]"
!pip install stable-baselines3[extra]

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,608 kB]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:6 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.1 MB]
Hit:10 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,862 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-b

In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from collections import deque
import gymnasium as gym
import matplotlib.pyplot as plt

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import BaseCallback


ENV_NAME = 'LunarLander-v3'

class Network(nn.Module):
    def __init__(self, state_size, action_size, seed=42):
        super(Network, self).__init__()
        self.seed = torch.manual_seed(seed)
        self.fc1 = nn.Linear(state_size, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, action_size)

    def forward(self, state):
        x = F.relu(self.fc1(state))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

class ReplayMemory(object):
    def __init__(self, capacity):
        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        self.capacity = capacity
        self.memory = deque(maxlen=capacity)

    def push(self, event):
        self.memory.append(event)

    def sample(self, batch_size):
        experiences = random.sample(self.memory, k=batch_size)
        states = torch.from_numpy(np.vstack([e[0] for e in experiences if e is not None])).float().to(self.device)
        actions = torch.from_numpy(np.vstack([e[1] for e in experiences if e is not None])).long().to(self.device)
        rewards = torch.from_numpy(np.vstack([e[2] for e in experiences if e is not None])).float().to(self.device)
        next_states = torch.from_numpy(np.vstack([e[3] for e in experiences if e is not None])).float().to(self.device)
        dones = torch.from_numpy(np.vstack([e[4] for e in experiences if e is not None]).astype(np.uint8)).float().to(self.device)
        return states, next_states, actions, rewards, dones

class Agent():
    def __init__(self, state_size, action_size):
        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        self.state_size = state_size
        self.action_size = action_size
        self.local_qnetwork = Network(state_size, action_size).to(self.device)
        self.target_qnetwork = Network(state_size, action_size).to(self.device)
        self.optimizer = optim.Adam(self.local_qnetwork.parameters(), lr=5e-4)
        self.memory = ReplayMemory(int(1e5))
        self.t_step = 0

    def step(self, state, action, reward, next_state, done):
        self.memory.push((state, action, reward, next_state, done))
        self.t_step = (self.t_step + 1) % 4
        if self.t_step == 0:
            if len(self.memory.memory) > 100:
                experiences = self.memory.sample(100)
                self.learn(experiences, 0.99)

    def act(self, state, epsilon=0.):
        state = torch.from_numpy(state).float().unsqueeze(0).to(self.device)
        self.local_qnetwork.eval()
        with torch.no_grad():
            action_values = self.local_qnetwork(state)
        self.local_qnetwork.train()
        if random.random() > epsilon:
            return np.argmax(action_values.cpu().data.numpy())
        else:
            return random.choice(np.arange(self.action_size))

    def learn(self, experiences, discount_factor):
        states, next_states, actions, rewards, dones = experiences
        next_q_targets = self.target_qnetwork(next_states).detach().max(1)[0].unsqueeze(1)
        q_targets = rewards + discount_factor * next_q_targets * (1 - dones)
        q_expected = self.local_qnetwork(states).gather(1, actions)
        loss = F.mse_loss(q_expected, q_targets)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        self.soft_update(self.local_qnetwork, self.target_qnetwork, 1e-3)

    def soft_update(self, local_model, target_model, interpolation_parameter):
        for target_param, local_param in zip(target_model.parameters(), local_model.parameters()):
            target_param.data.copy_(interpolation_parameter * local_param.data + (1.0 - interpolation_parameter) * target_param.data)

def train_dqn(env_name, num_episodes=1000):
    print("DQN training")
    env = gym.make(env_name)
    state_size = env.observation_space.shape[0]
    action_size = env.action_space.n
    agent = Agent(state_size, action_size)

    epsilon = 1.0
    scores_on_100_episodes = deque(maxlen=100)
    all_scores = []
    for episode in range(1, num_episodes + 1):
        state, _ = env.reset()
        score = 0
        for t in range(1000):
            action = agent.act(state, epsilon)
            next_state, reward, done, _, _ = env.step(action)
            agent.step(state, action, reward, next_state, done)
            state = next_state
            score += reward
            if done:
                break

        scores_on_100_episodes.append(score)
        all_scores.append(score)
        epsilon = max(0.01, 0.995 * epsilon)

        if episode % 100 == 0:
            print(f'\rEpisode {episode}\tAverage Score: {np.mean(scores_on_100_episodes):.2f}')

        if np.mean(scores_on_100_episodes) >= 200.0:
            print(f'\nEnvironment solved in {episode - 100:d} episodes!\tAverage Score: {np.mean(scores_on_100_episodes):.2f}')
            torch.save(agent.local_qnetwork.state_dict(), 'dqn_checkpoint.pth')
            break

    env.close()
    return all_scores

#ppo
class SaveRewardCallback(BaseCallback):
    def __init__(self, verbose=0):
        super(SaveRewardCallback, self).__init__(verbose)
        self.episode_rewards = []

    def _on_step(self) -> bool:
        if "episode" in self.locals["infos"][0]:
            self.episode_rewards.append(self.locals["infos"][0]["episode"]["r"])
        return True

def train_ppo(env_name, total_timesteps=300000):
    print("\nPPO Training---")
    env = gym.make(env_name)
    env = Monitor(env)

    model = PPO("MlpPolicy", env, verbose=0)

    reward_callback = SaveRewardCallback()
    model.learn(total_timesteps=total_timesteps, callback=reward_callback)

    model.save("ppo_lunar_lander")
    env.close()

    return reward_callback.episode_rewards


if __name__ == "__main__":
    dqn_scores = train_dqn(ENV_NAME, num_episodes=1000)
    ppo_scores = train_ppo(ENV_NAME, total_timesteps=300000)

    def moving_average(data, window_size=50):
        return np.convolve(data, np.ones(window_size)/window_size, mode='valid')

    dqn_ma = moving_average(dqn_scores)
    ppo_ma = moving_average(ppo_scores)

    plt.figure(figsize=(10, 6))
    plt.plot(dqn_ma, label='Custom DQN', alpha=0.8)
    plt.plot(ppo_ma, label='Stable Baselines3 PPO', alpha=0.8)
    plt.axhline(y=200, color='r', linestyle='--', label='Solved Threshold (200)')
    plt.title('Training Curve Comparison: DQN vs PPO on LunarLander-v3')
    plt.xlabel('Episodes')
    plt.ylabel('Average Reward (Moving Average)')
    plt.legend()
    plt.grid(True)
    plt.show()

    print("\n\n\nfinish\n.")

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type swigvarlink has no __module__ attribute


DQN training


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Episode 100	Average Score: -176.60


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Episode 200	Average Score: -103.49
Episode 300	Average Score: -59.16
Episode 400	Average Score: 19.87
Episode 500	Average Score: 126.98
Episode 600	Average Score: 179.24
Episode 700	Average Score: 180.49

Environment solved in 660 episodes!	Average Score: 203.14

PPO Training---


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


In [ ]:
import gymnasium as gym
import imageio
import torch
import base64
from IPython.display import HTML, display
from stable_baselines3 import PPO

def show_video_of_model(model_type, env_name="LunarLander-v3", filename="video.mp4"):
    env = gym.make(env_name, render_mode='rgb_array')
    state, _ = env.reset()
    frames = []
    done = False
    truncated = False

    if model_type == "DQN":
        state_size = env.observation_space.shape[0]
        action_size = env.action_space.n
        agent = Agent(state_size, action_size)
        agent.local_qnetwork.load_state_dict(torch.load('dqn_checkpoint.pth'))
        agent.local_qnetwork.eval()

    elif model_type == "PPO":
        model = PPO.load("ppo_lunar_lander")

    while not (done or truncated):
        frames.append(env.render())

        if model_type == "DQN":
            action = agent.act(state, epsilon=0.0)
        elif model_type == "PPO":
            action, _ = model.predict(state, deterministic=True)

        state, reward, done, truncated, _ = env.step(action)

    env.close()

    imageio.mimsave(filename, frames, fps=30)
    video = open(filename, 'r+b').read()
    encoded = base64.b64encode(video)
    display(HTML(data=f'''
        <video alt="test" autoplay loop controls style="height: 400px;">
            <source src="data:video/mp4;base64,{encoded.decode('ascii')}" type="video/mp4" />
        </video>
    '''))


show_video_of_model(model_type="DQN", filename="dqn_lunar_lander.mp4")



In [ ]:
show_video_of_model(model_type="PPO", filename="ppo_lunar_lander.mp4")